<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Chapter1_5_SubTest_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

รหัสนักศึกษา:6706021612143



ชื่อ-สกุล:นาย ณัฐกรณ์ มะลิซ้อน

# แบบทดสอบย่อยครั้งที่ 1
## 060243412 Advanced Database for Data Science
### ข้อสอบมีทั้งหมด 25 ข้อ รวม 25 คะแนน (หารเหลือ 10 คะแนน)

**โจทย์:** ณ ร้านกาแฟแห่งหนึ่ง ร้าน "Daily Brew" ต้องการวิเคราะห์ข้อมูลลูกค้า พนักงาน สินค้า และออเดอร์ ผ่าน SQL บน DuckDB


## โครงสร้างฐานข้อมูล (ER Overview)

ฐานข้อมูลร้านกาแฟ "Daily Brew" ประกอบด้วย 5 ตาราง ดังนี้

| ตาราง | คำอธิบาย | คีย์สำคัญ |
|---|---|---|
| `employee` | (พนักงาน) มีโครงสร้างลำดับชั้น ว่าพนักงานระดับใดรายงานใคร | `employee_id` PK, `manager_id` FK → `employee` (self-reference) |
| `customer` | (ลูกค้า) มีระบบแนะนำเพื่อน | `customer_id` PK, `referred_by` FK → `customer` (self-reference) |
| `product` | (เมนูสินค้า) แบ่งหมวดหมู่ กาแฟ/ชา/เบเกอรี่/อื่นๆ | `product_id` PK |
| `orders` | (ออเดอร์) 1 ออเดอร์ = 1 ครั้งที่ลูกค้าซื้อ | `order_id` PK, `customer_id` FK (NULL ได้ = walk-in), `employee_id` FK |
| `order_item` | (รายการสินค้าในแต่ละออเดอร์) many-to-many ระหว่าง orders กับ product | composite PK `(order_id, product_id)` |

**ความสัมพันธ์:**
* `orders` 1-to-many กับ `order_item`,
* `product` 1-to-many กับ `order_item`,
* `customer` 1-to-many กับ `orders` (แต่ `customer_id` เป็น NULL ได้สำหรับลูกค้า walk-in),
* `employee` 1-to-many กับ `orders`,
* และ `employee`/`customer` มี self-reference สำหรับลำดับชั้นผู้บังคับบัญชา และสายการแนะนำเพื่อนตามลำดับ


## คำอธิบาย Attribute แต่ละตาราง

### ตาราง `employee` (พนักงาน)

| Attribute | ชนิดข้อมูล | คำอธิบาย | Constraint / หมายเหตุ |
|---|---|---|---|
| `employee_id` | INTEGER | รหัสพนักงาน (surrogate key) | **PRIMARY KEY** |
| `name` | VARCHAR(100) | ชื่อ-นามสกุลพนักงาน | NOT NULL |
| `position` | VARCHAR(50) | ตำแหน่งงาน เช่น Area Manager, Branch Manager, Barista | NOT NULL |
| `branch` | VARCHAR(50) | สาขาที่พนักงานประจำอยู่ (สยาม / เอกมัย / รัชดา) | NOT NULL |
| `manager_id` | INTEGER | รหัสหัวหน้างานของพนักงานคนนี้ | **FOREIGN KEY** อ้างอิงตัวเอง (`employee.employee_id`) — self-reference สร้างโครงสร้างลำดับชั้น; เป็น `NULL` ได้สำหรับตำแหน่งสูงสุด (ไม่มีหัวหน้า) |
| `hire_date` | DATE | วันที่เริ่มงาน | NOT NULL |

### ตาราง `customer` (ลูกค้าสมาชิก)

| Attribute | ชนิดข้อมูล | คำอธิบาย | Constraint / หมายเหตุ |
|---|---|---|---|
| `customer_id` | INTEGER | รหัสลูกค้า (surrogate key) | **PRIMARY KEY** |
| `name` | VARCHAR(100) | ชื่อ-นามสกุลลูกค้า | NOT NULL |
| `city` | VARCHAR(50) | เมืองที่ลูกค้าอาศัยอยู่ | NOT NULL |
| `email` | VARCHAR(100) | อีเมลของลูกค้า | **UNIQUE** — ห้ามซ้ำกันระหว่างลูกค้า |
| `member_tier` | VARCHAR(10) | ระดับสมาชิก | **CHECK** ต้องเป็นค่าใดค่าหนึ่งใน `'bronze'`, `'silver'`, `'gold'` เท่านั้น |
| `join_date` | DATE | วันที่สมัครสมาชิก | NOT NULL |
| `referred_by` | INTEGER | รหัสลูกค้าที่เป็นผู้แนะนำ (referral) | **FOREIGN KEY** อ้างอิงตัวเอง (`customer.customer_id`) — self-reference สร้างสายการแนะนำเพื่อน; เป็น `NULL` ได้ถ้าไม่มีผู้แนะนำ |

### ตาราง `product` (เมนูสินค้า)

| Attribute | ชนิดข้อมูล | คำอธิบาย | Constraint / หมายเหตุ |
|---|---|---|---|
| `product_id` | INTEGER | รหัสสินค้า (surrogate key) | **PRIMARY KEY** |
| `product_name` | VARCHAR(100) | ชื่อสินค้า/เมนู | NOT NULL |
| `category` | VARCHAR(20) | หมวดหมู่สินค้า | **CHECK** ต้องเป็นค่าใดค่าหนึ่งใน `'กาแฟ'`, `'ชา'`, `'เบเกอรี่'`, `'อื่นๆ'` เท่านั้น |
| `price` | DECIMAL(6,2) | ราคาต่อหน่วย (บาท) | NOT NULL, **CHECK** ต้องมากกว่า 0 |
| `is_seasonal` | BOOLEAN | สินค้าตามฤดูกาลหรือไม่ | ค่าเริ่มต้น (`DEFAULT`) คือ `FALSE` |

### ตาราง `orders` (หัวออเดอร์)

| Attribute | ชนิดข้อมูล | คำอธิบาย | Constraint / หมายเหตุ |
|---|---|---|---|
| `order_id` | INTEGER | รหัสออเดอร์ (surrogate key) | **PRIMARY KEY** |
| `customer_id` | INTEGER | รหัสลูกค้าที่สั่งซื้อ | **FOREIGN KEY** อ้างอิง `customer.customer_id`; เป็น `NULL` ได้ — หมายถึงลูกค้า walk-in ที่ไม่มีบัตรสมาชิก |
| `employee_id` | INTEGER | รหัสพนักงานที่รับออเดอร์ | **FOREIGN KEY** อ้างอิง `employee.employee_id`; **NOT NULL** — ทุกออเดอร์ต้องมีพนักงานรับ |
| `branch` | VARCHAR(50) | สาขาที่เกิดออเดอร์นี้ | NOT NULL |
| `order_date` | DATE | วันที่ทำรายการ | NOT NULL |
| `order_channel` | VARCHAR(10) | ช่องทางการสั่งซื้อ | **CHECK** ต้องเป็นค่าใดค่าหนึ่งใน `'walk-in'`, `'delivery'`, `'pickup'` เท่านั้น |

### ตาราง `order_item` (รายการสินค้าในออเดอร์)

| Attribute | ชนิดข้อมูล | คำอธิบาย | Constraint / หมายเหตุ |
|---|---|---|---|
| `order_id` | INTEGER | รหัสออเดอร์ที่รายการนี้สังกัดอยู่ | **FOREIGN KEY** อ้างอิง `orders.order_id` และเป็นส่วนหนึ่งของ **composite PRIMARY KEY** |
| `product_id` | INTEGER | รหัสสินค้าที่ถูกสั่งในรายการนี้ | **FOREIGN KEY** อ้างอิง `product.product_id` และเป็นส่วนหนึ่งของ **composite PRIMARY KEY** |
| `quantity` | INTEGER | จำนวนที่สั่งของสินค้าชิ้นนั้นในออเดอร์นี้ | NOT NULL, **CHECK** ต้องมากกว่า 0 |
| `unit_price` | DECIMAL(6,2) | ราคาต่อหน่วย ณ เวลาที่สั่งซื้อ | NOT NULL — เก็บแยกจาก `product.price` เพื่อรักษาราคาที่แท้จริงตอนขาย แม้ราคาสินค้าจะเปลี่ยนในภายหลัง |


## Setup ฐานข้อมูลเริ่มต้น (รันเซลล์เหล่านี้ก่อนเสมอ)

In [1]:
!pip install duckdb --quiet

In [2]:
import duckdb

con = duckdb.connect(database=":memory:")

def run(sql):
    """helper: รัน SQL แล้วคืนผลลัพธ์เป็น pandas DataFrame"""
    return con.execute(sql).df()


### สร้างโครงสร้างตาราง

In [3]:
con.execute("""
CREATE TABLE employee (
    employee_id  INTEGER PRIMARY KEY,
    name         VARCHAR(100) NOT NULL,
    position     VARCHAR(50) NOT NULL,
    branch       VARCHAR(50) NOT NULL,
    manager_id   INTEGER REFERENCES employee(employee_id),
    hire_date    DATE NOT NULL
);

CREATE TABLE customer (
    customer_id   INTEGER PRIMARY KEY,
    name          VARCHAR(100) NOT NULL,
    city          VARCHAR(50) NOT NULL,
    email         VARCHAR(100) UNIQUE,
    member_tier   VARCHAR(10) CHECK (member_tier IN ('bronze','silver','gold')),
    join_date     DATE NOT NULL,
    referred_by   INTEGER REFERENCES customer(customer_id)
);

CREATE TABLE product (
    product_id    INTEGER PRIMARY KEY,
    product_name  VARCHAR(100) NOT NULL,
    category      VARCHAR(20) CHECK (category IN ('กาแฟ','ชา','เบเกอรี่','อื่นๆ')),
    price         DECIMAL(6,2) NOT NULL CHECK (price > 0),
    is_seasonal   BOOLEAN DEFAULT FALSE
);

CREATE TABLE orders (
    order_id       INTEGER PRIMARY KEY,
    customer_id    INTEGER REFERENCES customer(customer_id),
    employee_id    INTEGER NOT NULL REFERENCES employee(employee_id),
    branch         VARCHAR(50) NOT NULL,
    order_date     DATE NOT NULL,
    order_channel  VARCHAR(10) CHECK (order_channel IN ('walk-in','delivery','pickup'))
);

CREATE TABLE order_item (
    order_id     INTEGER REFERENCES orders(order_id),
    product_id   INTEGER REFERENCES product(product_id),
    quantity     INTEGER NOT NULL CHECK (quantity > 0),
    unit_price   DECIMAL(6,2) NOT NULL,
    PRIMARY KEY (order_id, product_id)
);
""")


### เพิ่มข้อมูลเริ่มต้น

In [4]:
con.execute("""
-- employee
INSERT INTO employee VALUES (1, 'สมชาย ผู้จัดการเขต', 'Area Manager', 'สยาม', NULL, '2023-01-05');
INSERT INTO employee VALUES (2, 'สุนีย์ หัวหน้าสาขา', 'Branch Manager', 'สยาม', 1, '2023-02-01');
INSERT INTO employee VALUES (3, 'ประวิทย์ หัวหน้าสาขา', 'Branch Manager', 'เอกมัย', 1, '2023-02-10');
INSERT INTO employee VALUES (4, 'อรทัย บาริสต้า', 'Barista', 'สยาม', 2, '2023-03-01');
INSERT INTO employee VALUES (5, 'ธนกร บาริสต้า', 'Barista', 'สยาม', 2, '2023-03-15');
INSERT INTO employee VALUES (6, 'ปิยะดา บาริสต้า', 'Barista', 'สยาม', 2, '2023-04-01');
INSERT INTO employee VALUES (7, 'วีระชัย บาริสต้า', 'Barista', 'เอกมัย', 3, '2023-03-05');
INSERT INTO employee VALUES (8, 'ศศิธร บาริสต้า', 'Barista', 'เอกมัย', 3, '2023-04-20');
INSERT INTO employee VALUES (9, 'กิตติ บาริสต้า', 'Barista', 'เอกมัย', 3, '2023-05-01');
INSERT INTO employee VALUES (10, 'มณีรัตน์ บาริสต้า', 'Barista', 'รัชดา', 1, '2023-06-01');

-- customer
INSERT INTO customer VALUES (1, 'กมล ศรีสุข', 'กรุงเทพฯ', 'kamol@mail.com', 'gold', '2023-01-10', NULL);
INSERT INTO customer VALUES (2, 'สุดา วงศ์ทอง', 'กรุงเทพฯ', 'suda@mail.com', 'gold', '2023-01-15', NULL);
INSERT INTO customer VALUES (3, 'วิชัย เพชรดี', 'นนทบุรี', 'wichai@mail.com', 'silver', '2023-02-01', 1);
INSERT INTO customer VALUES (4, 'มาลี บุญมาก', 'กรุงเทพฯ', 'malee@mail.com', 'silver', '2023-02-05', 1);
INSERT INTO customer VALUES (5, 'ประเสริฐ ทองแท้', 'กรุงเทพฯ', 'prasert@mail.com', 'bronze', '2023-02-20', 3);
INSERT INTO customer VALUES (6, 'อรุณี แสงทอง', 'ปทุมธานี', 'arunee@mail.com', 'silver', '2023-03-01', 2);
INSERT INTO customer VALUES (7, 'ธีระพล มั่นคง', 'กรุงเทพฯ', 'teerapon@mail.com', 'bronze', '2023-03-10', NULL);
INSERT INTO customer VALUES (8, 'นภาพร ใจดี', 'นนทบุรี', 'napaporn@mail.com', 'bronze', '2023-03-15', 5);
INSERT INTO customer VALUES (9, 'ชัยวัฒน์ รุ่งเรือง', 'กรุงเทพฯ', 'chaiwat@mail.com', 'gold', '2023-04-01', NULL);
INSERT INTO customer VALUES (10, 'ปราณี สุขใจ', 'กรุงเทพฯ', 'pranee@mail.com', 'silver', '2023-04-05', 9);
INSERT INTO customer VALUES (11, 'สมพงษ์ ดีเลิศ', 'ปทุมธานี', 'sompong@mail.com', 'bronze', '2023-04-10', 6);
INSERT INTO customer VALUES (12, 'รัตนา เกียรติยศ', 'กรุงเทพฯ', 'rattana@mail.com', 'bronze', '2023-04-20', NULL);
INSERT INTO customer VALUES (13, 'อนุชา พูลสวัสดิ์', 'นนทบุรี', 'anucha@mail.com', 'silver', '2023-05-01', 8);
INSERT INTO customer VALUES (14, 'จินตนา ศรีวิไล', 'กรุงเทพฯ', 'jintana@mail.com', 'bronze', '2023-05-10', NULL);
INSERT INTO customer VALUES (15, 'ณัฐพล วัฒนกุล', 'กรุงเทพฯ', 'nattapon@mail.com', 'bronze', '2023-05-15', 9);
INSERT INTO customer VALUES (16, 'พิมพ์ใจ อ่อนละมัย', 'กรุงเทพฯ', 'pimjai@mail.com', 'bronze', '2023-06-01', NULL);
INSERT INTO customer VALUES (17, 'สายฝน ชื่นใจ', 'ปทุมธานี', 'saifon@mail.com', 'bronze', '2023-06-10', 11);
INSERT INTO customer VALUES (18, 'อัครเดช ยิ่งยง', 'กรุงเทพฯ', 'akaradet@mail.com', 'silver', '2023-06-15', NULL);
INSERT INTO customer VALUES (19, 'ดวงใจ แก้วมณี', 'นนทบุรี', 'duangjai@mail.com', 'bronze', '2023-07-01', 13);
INSERT INTO customer VALUES (20, 'ไพโรจน์ เจริญสุข', 'กรุงเทพฯ', 'pairoj@mail.com', 'bronze', '2023-07-05', NULL);

-- product
INSERT INTO product VALUES (1, 'เอสเปรสโซ่', 'กาแฟ', 65.0, FALSE);
INSERT INTO product VALUES (2, 'อเมริกาโน่', 'กาแฟ', 70.0, FALSE);
INSERT INTO product VALUES (3, 'ลาเต้', 'กาแฟ', 80.0, FALSE);
INSERT INTO product VALUES (4, 'คาปูชิโน่', 'กาแฟ', 80.0, FALSE);
INSERT INTO product VALUES (5, 'มอคค่า', 'กาแฟ', 90.0, FALSE);
INSERT INTO product VALUES (6, 'กาแฟเย็นส้มเกลี้ยง', 'กาแฟ', 95.0, TRUE);
INSERT INTO product VALUES (7, 'ชาไทย', 'ชา', 60.0, FALSE);
INSERT INTO product VALUES (8, 'ชาเขียวมัทฉะ', 'ชา', 85.0, FALSE);
INSERT INTO product VALUES (9, 'ชามะนาว', 'ชา', 55.0, FALSE);
INSERT INTO product VALUES (10, 'ชาดอกเก๊กฮวยเย็น', 'ชา', 60.0, TRUE);
INSERT INTO product VALUES (11, 'ครัวซองต์เนย', 'เบเกอรี่', 55.0, FALSE);
INSERT INTO product VALUES (12, 'บราวนี่ช็อกโกแลต', 'เบเกอรี่', 65.0, FALSE);
INSERT INTO product VALUES (13, 'เค้กมะพร้าวอ่อน', 'เบเกอรี่', 75.0, TRUE);
INSERT INTO product VALUES (14, 'แซนด์วิชแฮมชีส', 'อื่นๆ', 70.0, FALSE);
INSERT INTO product VALUES (15, 'น้ำเปล่า', 'อื่นๆ', 20.0, FALSE);

-- orders
INSERT INTO orders VALUES (1, 1, 4, 'สยาม', '2024-02-03', 'walk-in');
INSERT INTO orders VALUES (2, 2, 4, 'สยาม', '2024-02-05', 'pickup');
INSERT INTO orders VALUES (3, NULL, 5, 'สยาม', '2024-02-06', 'walk-in');
INSERT INTO orders VALUES (4, 3, 7, 'เอกมัย', '2024-02-08', 'delivery');
INSERT INTO orders VALUES (5, 4, 4, 'สยาม', '2024-02-10', 'walk-in');
INSERT INTO orders VALUES (6, 1, 6, 'สยาม', '2024-02-14', 'pickup');
INSERT INTO orders VALUES (7, 6, 7, 'เอกมัย', '2024-02-15', 'walk-in');
INSERT INTO orders VALUES (8, NULL, 8, 'เอกมัย', '2024-02-18', 'walk-in');
INSERT INTO orders VALUES (9, 2, 5, 'สยาม', '2024-02-20', 'delivery');
INSERT INTO orders VALUES (10, 9, 4, 'สยาม', '2024-02-25', 'walk-in');
INSERT INTO orders VALUES (11, 1, 4, 'สยาม', '2024-03-02', 'walk-in');
INSERT INTO orders VALUES (12, 3, 7, 'เอกมัย', '2024-03-04', 'walk-in');
INSERT INTO orders VALUES (13, 7, 6, 'สยาม', '2024-03-06', 'pickup');
INSERT INTO orders VALUES (14, 9, 5, 'สยาม', '2024-03-09', 'delivery');
INSERT INTO orders VALUES (15, 2, 4, 'สยาม', '2024-03-12', 'walk-in');
INSERT INTO orders VALUES (16, NULL, 9, 'เอกมัย', '2024-03-14', 'walk-in');
INSERT INTO orders VALUES (17, 12, 10, 'รัชดา', '2024-03-16', 'walk-in');
INSERT INTO orders VALUES (18, 6, 8, 'เอกมัย', '2024-03-18', 'delivery');
INSERT INTO orders VALUES (19, 1, 6, 'สยาม', '2024-03-21', 'pickup');
INSERT INTO orders VALUES (20, 9, 4, 'สยาม', '2024-03-25', 'walk-in');
INSERT INTO orders VALUES (21, 14, 6, 'สยาม', '2024-04-01', 'walk-in');
INSERT INTO orders VALUES (22, 3, 7, 'เอกมัย', '2024-04-03', 'walk-in');
INSERT INTO orders VALUES (23, 2, 5, 'สยาม', '2024-04-06', 'delivery');
INSERT INTO orders VALUES (24, 18, 10, 'รัชดา', '2024-04-08', 'walk-in');
INSERT INTO orders VALUES (25, 9, 4, 'สยาม', '2024-04-11', 'pickup');
INSERT INTO orders VALUES (26, 7, 6, 'สยาม', '2024-04-14', 'walk-in');
INSERT INTO orders VALUES (27, NULL, 9, 'เอกมัย', '2024-04-16', 'walk-in');
INSERT INTO orders VALUES (28, 1, 4, 'สยาม', '2024-04-19', 'walk-in');
INSERT INTO orders VALUES (29, 12, 10, 'รัชดา', '2024-04-22', 'walk-in');
INSERT INTO orders VALUES (30, 20, 6, 'สยาม', '2024-04-27', 'delivery');
INSERT INTO orders VALUES (31, 2, 5, 'สยาม', '2024-05-02', 'walk-in');
INSERT INTO orders VALUES (32, 9, 4, 'สยาม', '2024-05-05', 'walk-in');
INSERT INTO orders VALUES (33, 6, 7, 'เอกมัย', '2024-05-08', 'pickup');
INSERT INTO orders VALUES (34, 14, 6, 'สยาม', '2024-05-10', 'walk-in');
INSERT INTO orders VALUES (35, 18, 10, 'รัชดา', '2024-05-13', 'walk-in');
INSERT INTO orders VALUES (36, 1, 4, 'สยาม', '2024-05-17', 'delivery');
INSERT INTO orders VALUES (37, 3, 8, 'เอกมัย', '2024-05-20', 'walk-in');
INSERT INTO orders VALUES (38, NULL, 5, 'สยาม', '2024-05-23', 'walk-in');
INSERT INTO orders VALUES (39, 9, 4, 'สยาม', '2024-05-27', 'pickup');
INSERT INTO orders VALUES (40, 12, 10, 'รัชดา', '2024-05-30', 'walk-in');
INSERT INTO orders VALUES (41, 2, 4, 'สยาม', '2024-06-02', 'walk-in');
INSERT INTO orders VALUES (42, 7, 6, 'สยาม', '2024-06-05', 'walk-in');
INSERT INTO orders VALUES (43, 9, 5, 'สยาม', '2024-06-09', 'delivery');
INSERT INTO orders VALUES (44, 18, 10, 'รัชดา', '2024-06-12', 'walk-in');
INSERT INTO orders VALUES (45, 3, 7, 'เอกมัย', '2024-06-15', 'walk-in');
INSERT INTO orders VALUES (46, 1, 4, 'สยาม', '2024-06-19', 'pickup');
INSERT INTO orders VALUES (47, 9, 4, 'สยาม', '2024-06-23', 'walk-in');
INSERT INTO orders VALUES (48, 6, 9, 'เอกมัย', '2024-06-27', 'walk-in');
INSERT INTO orders VALUES (49, 2, 5, 'สยาม', '2024-07-01', 'delivery');
INSERT INTO orders VALUES (50, 9, 4, 'สยาม', '2024-07-05', 'walk-in');

-- order_item
INSERT INTO order_item VALUES (1, 1, 1, 65.0);
INSERT INTO order_item VALUES (1, 11, 1, 55.0);
INSERT INTO order_item VALUES (2, 3, 2, 80.0);
INSERT INTO order_item VALUES (3, 2, 1, 70.0);
INSERT INTO order_item VALUES (3, 15, 1, 20.0);
INSERT INTO order_item VALUES (4, 4, 1, 80.0);
INSERT INTO order_item VALUES (4, 12, 1, 65.0);
INSERT INTO order_item VALUES (5, 3, 1, 80.0);
INSERT INTO order_item VALUES (5, 7, 1, 60.0);
INSERT INTO order_item VALUES (6, 1, 2, 65.0);
INSERT INTO order_item VALUES (6, 11, 2, 55.0);
INSERT INTO order_item VALUES (7, 7, 2, 60.0);
INSERT INTO order_item VALUES (8, 9, 1, 55.0);
INSERT INTO order_item VALUES (9, 5, 1, 90.0);
INSERT INTO order_item VALUES (9, 13, 1, 75.0);
INSERT INTO order_item VALUES (10, 3, 3, 80.0);
INSERT INTO order_item VALUES (10, 12, 1, 65.0);
INSERT INTO order_item VALUES (11, 1, 1, 65.0);
INSERT INTO order_item VALUES (12, 4, 2, 80.0);
INSERT INTO order_item VALUES (13, 7, 1, 60.0);
INSERT INTO order_item VALUES (13, 14, 1, 70.0);
INSERT INTO order_item VALUES (14, 5, 2, 90.0);
INSERT INTO order_item VALUES (15, 3, 1, 80.0);
INSERT INTO order_item VALUES (15, 11, 1, 55.0);
INSERT INTO order_item VALUES (16, 9, 1, 55.0);
INSERT INTO order_item VALUES (16, 15, 1, 20.0);
INSERT INTO order_item VALUES (17, 2, 1, 70.0);
INSERT INTO order_item VALUES (18, 6, 1, 95.0);
INSERT INTO order_item VALUES (18, 13, 1, 75.0);
INSERT INTO order_item VALUES (19, 1, 1, 65.0);
INSERT INTO order_item VALUES (19, 11, 1, 55.0);
INSERT INTO order_item VALUES (20, 3, 2, 80.0);
INSERT INTO order_item VALUES (21, 7, 1, 60.0);
INSERT INTO order_item VALUES (22, 4, 1, 80.0);
INSERT INTO order_item VALUES (22, 12, 1, 65.0);
INSERT INTO order_item VALUES (23, 5, 1, 90.0);
INSERT INTO order_item VALUES (24, 2, 2, 70.0);
INSERT INTO order_item VALUES (25, 3, 1, 80.0);
INSERT INTO order_item VALUES (26, 7, 2, 60.0);
INSERT INTO order_item VALUES (26, 14, 1, 70.0);
INSERT INTO order_item VALUES (27, 10, 1, 60.0);
INSERT INTO order_item VALUES (28, 1, 3, 65.0);
INSERT INTO order_item VALUES (29, 4, 1, 80.0);
INSERT INTO order_item VALUES (30, 6, 1, 95.0);
INSERT INTO order_item VALUES (30, 13, 1, 75.0);
INSERT INTO order_item VALUES (31, 5, 1, 90.0);
INSERT INTO order_item VALUES (31, 13, 1, 75.0);
INSERT INTO order_item VALUES (32, 3, 2, 80.0);
INSERT INTO order_item VALUES (33, 7, 1, 60.0);
INSERT INTO order_item VALUES (34, 7, 1, 60.0);
INSERT INTO order_item VALUES (34, 14, 1, 70.0);
INSERT INTO order_item VALUES (35, 2, 1, 70.0);
INSERT INTO order_item VALUES (36, 1, 1, 65.0);
INSERT INTO order_item VALUES (36, 11, 2, 55.0);
INSERT INTO order_item VALUES (37, 8, 1, 85.0);
INSERT INTO order_item VALUES (38, 2, 1, 70.0);
INSERT INTO order_item VALUES (39, 3, 1, 80.0);
INSERT INTO order_item VALUES (40, 4, 2, 80.0);
INSERT INTO order_item VALUES (41, 3, 1, 80.0);
INSERT INTO order_item VALUES (41, 11, 1, 55.0);
INSERT INTO order_item VALUES (42, 7, 1, 60.0);
INSERT INTO order_item VALUES (43, 5, 2, 90.0);
INSERT INTO order_item VALUES (43, 13, 1, 75.0);
INSERT INTO order_item VALUES (44, 2, 1, 70.0);
INSERT INTO order_item VALUES (45, 4, 1, 80.0);
INSERT INTO order_item VALUES (45, 12, 1, 65.0);
INSERT INTO order_item VALUES (46, 1, 2, 65.0);
INSERT INTO order_item VALUES (47, 3, 1, 80.0);
INSERT INTO order_item VALUES (48, 8, 1, 85.0);
INSERT INTO order_item VALUES (48, 10, 1, 60.0);
INSERT INTO order_item VALUES (49, 5, 1, 90.0);
INSERT INTO order_item VALUES (50, 3, 3, 80.0);
INSERT INTO order_item VALUES (50, 12, 1, 65.0);
""")


In [5]:
# ตรวจสอบว่าข้อมูลเริ่มต้นถูกโหลดครบถ้วน
for t in ["employee", "customer", "product", "orders", "order_item"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} แถว")


employee: 10 แถว
customer: 20 แถว
product: 15 แถว
orders: 50 แถว
order_item: 73 แถว


---
## SQL Selection & Complex Conditions

### ข้อที่ 1: SELECT / ORDER BY / LIMIT

จงเขียน query แสดงสินค้า 5 อันดับที่ **ราคาแพงที่สุด** โดยเรียงจากแพงไปถูก

In [8]:
#เขียนข้อ 1 โค้ดตรงนี้
run("""
  SELECT product_name, category, price FROM product ORDER BY price DESC LIMIT 5
""")

,product_name,category,price
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ,95.0
1,มอคค่า,กาแฟ,90.0
2,ชาเขียวมัทฉะ,ชา,85.0
3,ลาเต้,กาแฟ,80.0
4,คาปูชิโน่,กาแฟ,80.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 1

,product_name,category,price
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ,95.0
1,มอคค่า,กาแฟ,90.0
2,ชาเขียวมัทฉะ,ชา,85.0
3,ลาเต้,กาแฟ,80.0
4,คาปูชิโน่,กาแฟ,80.0


### ข้อที่ 2: AND/OR/NOT Precedence

ทางร้านต้องการรายชื่อสินค้าที่**เป็นหมวดกาแฟและราคาตั้งแต่ 80 บาทขึ้นไป หรือเป็นสินค้าตามฤดูกาล** ให้เรียงตาม `category`ก-ฮ, `price`แพงไปถูก

In [11]:
#เขียนข้อ 2 โค้ดตรงนี้
run("""
  SELECT product_name, category, price, is_seasonal FROM product
  WHERE (category = 'กาแฟ' AND price >= 80) OR is_seasonal = 1
  ORDER BY category ASC, price DESC;
""")

,product_name,category,price,is_seasonal
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ,95.0,True
1,มอคค่า,กาแฟ,90.0,False
2,ลาเต้,กาแฟ,80.0,False
3,คาปูชิโน่,กาแฟ,80.0,False
4,ชาดอกเก๊กฮวยเย็น,ชา,60.0,True
5,เค้กมะพร้าวอ่อน,เบเกอรี่,75.0,True


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 2

,product_name,category,price,is_seasonal
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ,95.0,True
1,มอคค่า,กาแฟ,90.0,False
2,ลาเต้,กาแฟ,80.0,False
3,คาปูชิโน่,กาแฟ,80.0,False
4,ชาดอกเก๊กฮวยเย็น,ชา,60.0,True
5,เค้กมะพร้าวอ่อน,เบเกอรี่,75.0,True


### ข้อที่ 3: AND/OR/NOT Precedence

ทางร้านต้องการรายชื่อสินค้าที่**เป็นหมวดชาหรือว่าเป็นสินค้าไม่ตรงตามฤดูกาลก็ได้ แต่ราคาต้องมากกว่า 80 บาทขึ้นไป** ให้เรียงตาม `category`ก-ฮ, `price`ถูกไปแพง

In [24]:
#เขียนข้อ 3 โค้ดตรงนี้
run("""
    SELECT product_name, category, price, is_seasonal FROM product
    WHERE (category LIKE '%ชา%' OR category LIKE '%กาแฟ%') AND price >= 80 AND is_seasonal = 0
    ORDER BY category ASC, price ASC


""")

,product_name,category,price,is_seasonal
0,ลาเต้,กาแฟ,80.0,False
1,คาปูชิโน่,กาแฟ,80.0,False
2,มอคค่า,กาแฟ,90.0,False
3,ชาเขียวมัทฉะ,ชา,85.0,False


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 3

,product_name,category,price,is_seasonal
0,ลาเต้,กาแฟ,80.0,False
1,คาปูชิโน่,กาแฟ,80.0,False
2,มอคค่า,กาแฟ,90.0,False
3,ชาเขียวมัทฉะ,ชา,85.0,False


### ข้อที่ 4: BETWEEN + IN

จงเขียน query หาสินค้าในหมวด **"กาแฟ" หรือ "ชา"** ที่มีราคาอยู่ในช่วง **60-85 บาท** โดยใช้ `BETWEEN` และ `IN`

In [21]:
#เขียนข้อ 4 โค้ดตรงนี้
run("""
  SELECT product_name, category,  price FROM product
  WHERE category IN ('กาแฟ', 'ชา') AND price BETWEEN 60 AND 85
  ORDER BY price ASC
""")

,product_name,category,price
0,ชาไทย,ชา,60.0
1,ชาดอกเก๊กฮวยเย็น,ชา,60.0
2,เอสเปรสโซ่,กาแฟ,65.0
3,อเมริกาโน่,กาแฟ,70.0
4,ลาเต้,กาแฟ,80.0
5,คาปูชิโน่,กาแฟ,80.0
6,ชาเขียวมัทฉะ,ชา,85.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 4

,product_name,category,price
0,ชาไทย,ชา,60.0
1,ชาดอกเก๊กฮวยเย็น,ชา,60.0
2,เอสเปรสโซ่,กาแฟ,65.0
3,อเมริกาโน่,กาแฟ,70.0
4,ลาเต้,กาแฟ,80.0
5,คาปูชิโน่,กาแฟ,80.0
6,ชาเขียวมัทฉะ,ชา,85.0


### ข้อที่ 5: LIKE Pattern Matching

จงเขียน query ค้นหาสินค้าที่มีคำว่า **"เย็น"** อยู่ในชื่อสินค้า (`product_name`) โดยใช้ `LIKE`

In [25]:
#เขียนข้อ 5 โค้ดตรงนี้
run("""
  SELECT product_name, category FROM product
  WHERE product_name LIKE '%เย็น%';

""")

,product_name,category
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ
1,ชาดอกเก๊กฮวยเย็น,ชา


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 5

,product_name,category
0,กาแฟเย็นส้มเกลี้ยง,กาแฟ
1,ชาดอกเก๊กฮวยเย็น,ชา


### ข้อที่ 6: NULL Handling

ทางร้านต้องการรู้ว่ามีรายการใดบ้างที่มี**ออเดอร์ walk-in ที่ไม่มีบัตรสมาชิก** (คือ `customer_id` เป็น NULL) ให้เรียงตาม branch ก-ฮ


In [27]:
#เขียนข้อ 6 โค้ดตรงนี้
run("""
    SELECT order_id,	employee_id,	customer_id, branch, order_date FROM  orders
    WHERE order_id NOT NULL AND customer_id IS NULL
    ORDER BY branch ASC
""")

,order_id,employee_id,customer_id,branch,order_date
0,3,5,<NA>,สยาม,2024-02-06
1,38,5,<NA>,สยาม,2024-05-23
2,8,8,<NA>,เอกมัย,2024-02-18
3,16,9,<NA>,เอกมัย,2024-03-14
4,27,9,<NA>,เอกมัย,2024-04-16


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 6

,order_id,employee_id,customer_id,branch,order_date
0,3,5,<NA>,สยาม,2024-02-06
1,38,5,<NA>,สยาม,2024-05-23
2,8,8,<NA>,เอกมัย,2024-02-18
3,16,9,<NA>,เอกมัย,2024-03-14
4,27,9,<NA>,เอกมัย,2024-04-16


### ข้อที่ 7: CASE + Sargability

จงเขียน query แสดงชื่อสินค้า ราคา และป้ายราคา (`price_band`) ให้แบ่งด้วย `CASE WHEN` โดยที่
   - ถ้าราคา < 60 ให้เป็น `'ประหยัด'`
   - ถ้าราคาอยู่ระหว่าง 60-85 ให้เป็น `'มาตรฐาน'`
   - ถ้าราคา > 85 ให้เป็น `'พรีเมียม'`

ให้เรียงข้อมูลตาม `price_band` และ `price`

In [29]:
#เขียนข้อ 7 โค้ดตรงนี้
run("""
    SELECT product_name, price,
        CASE
            WHEN price < 60 THEN 'ประหยัด'
            WHEN price BETWEEN 60 AND 85 THEN 'มาตรฐาน'
            ELSE 'พรีเมียม'
        END AS price_band
    FROM product
    ORDER BY price_band ASC

""")

,product_name,price,price_band
0,ชามะนาว,55.0,ประหยัด
1,ครัวซองต์เนย,55.0,ประหยัด
2,น้ำเปล่า,20.0,ประหยัด
3,มอคค่า,90.0,พรีเมียม
4,กาแฟเย็นส้มเกลี้ยง,95.0,พรีเมียม
5,เอสเปรสโซ่,65.0,มาตรฐาน
6,อเมริกาโน่,70.0,มาตรฐาน
7,ลาเต้,80.0,มาตรฐาน
8,คาปูชิโน่,80.0,มาตรฐาน
9,ชาไทย,60.0,มาตรฐาน


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 7

,product_name,price,price_band
0,น้ำเปล่า,20.0,ประหยัด
1,ชามะนาว,55.0,ประหยัด
2,ครัวซองต์เนย,55.0,ประหยัด
3,มอคค่า,90.0,พรีเมียม
4,กาแฟเย็นส้มเกลี้ยง,95.0,พรีเมียม
5,ชาไทย,60.0,มาตรฐาน
6,ชาดอกเก๊กฮวยเย็น,60.0,มาตรฐาน
7,เอสเปรสโซ่,65.0,มาตรฐาน
8,บราวนี่ช็อกโกแลต,65.0,มาตรฐาน
9,อเมริกาโน่,70.0,มาตรฐาน


---
## Joins & Join Algorithms

### ข้อที่ 8: INNER JOIN หลายตาราง

จงเขียน query แสดง **ยอดรวมของแต่ละออเดอร์** (`order_id`, ชื่อลูกค้า, ยอดรวม) โดย JOIN ตาราง `orders`, `customer`, `order_item` เข้าด้วยกัน (แสดงเฉพาะ 10 แถวแรก เรียงตาม order_id)

In [33]:
run("DESCRIBE order_item")

,column_name,column_type,null,key,default,extra
0,order_id,INTEGER,NO,PRI,None,None
1,product_id,INTEGER,NO,PRI,None,None
2,quantity,INTEGER,NO,None,None,None
3,unit_price,"DECIMAL(6,2)",NO,None,None,None


In [37]:
#เขียนข้อ 8 โค้ดตรงนี้
run("""
SELECT o.order_id, c.name, SUM(oi.quantity * p.price) AS total_amount
FROM orders o
JOIN
    customer c ON o.customer_id = c.customer_id
JOIN
    order_item oi ON o.order_id = oi.order_id
JOIN
    product p ON oi.product_id = p.product_id
GROUP BY
    o.order_id,
    c.name
ORDER BY
    o.order_id ASC
LIMIT 10;

""")

,order_id,name,total_amount
0,1,กมล ศรีสุข,120.0
1,2,สุดา วงศ์ทอง,160.0
2,4,วิชัย เพชรดี,145.0
3,5,มาลี บุญมาก,140.0
4,6,กมล ศรีสุข,240.0
5,7,อรุณี แสงทอง,120.0
6,9,สุดา วงศ์ทอง,165.0
7,10,ชัยวัฒน์ รุ่งเรือง,305.0
8,11,กมล ศรีสุข,65.0
9,12,วิชัย เพชรดี,160.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 8

,order_id,customer_name,order_total
0,1,กมล ศรีสุข,120.0
1,2,สุดา วงศ์ทอง,160.0
2,4,วิชัย เพชรดี,145.0
3,5,มาลี บุญมาก,140.0
4,6,กมล ศรีสุข,240.0
5,7,อรุณี แสงทอง,120.0
6,9,สุดา วงศ์ทอง,165.0
7,10,ชัยวัฒน์ รุ่งเรือง,305.0
8,11,กมล ศรีสุข,65.0
9,12,วิชัย เพชรดี,160.0


### ข้อที่ 9: LEFT JOIN + IS NULL

จงเขียน query หา **"ลูกค้าที่ยังไม่เคยสั่งซื้อเลยสักครั้ง"** โดยใช้ `LEFT JOIN` ร่วมกับ `IS NULL` จัดเรียงตามชื่อลูกค้า ก-ฮ

In [41]:
#เขียนข้อ 9 โค้ดตรงนี้
run("""
    SELECT c.customer_id, c.name
    FROM customer c
    LEFT JOIN orders AS o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
    ORDER BY c.name ASC;

""")

,customer_id,name
0,15,ณัฐพล วัฒนกุล
1,19,ดวงใจ แก้วมณี
2,8,นภาพร ใจดี
3,5,ประเสริฐ ทองแท้
4,10,ปราณี สุขใจ
5,16,พิมพ์ใจ อ่อนละมัย
6,11,สมพงษ์ ดีเลิศ
7,17,สายฝน ชื่นใจ
8,13,อนุชา พูลสวัสดิ์


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 9

,customer_id,name
0,15,ณัฐพล วัฒนกุล
1,19,ดวงใจ แก้วมณี
2,8,นภาพร ใจดี
3,5,ประเสริฐ ทองแท้
4,10,ปราณี สุขใจ
5,16,พิมพ์ใจ อ่อนละมัย
6,11,สมพงษ์ ดีเลิศ
7,17,สายฝน ชื่นใจ
8,13,อนุชา พูลสวัสดิ์


### ข้อที่ 10: RIGHT JOIN

จากข้อที่แล้ว ให้เขียน query หา **"ลูกค้าที่ยังไม่เคยสั่งซื้อเลยสักครั้ง"** แบบเดิมอีกครั้ง แต่คราวนี้ใช้ `RIGHT JOIN` แทน

In [48]:
#เขียนข้อ 10 โค้ดตรงนี้
run("""
        SELECT c.customer_id, c.name
        FROM orders o
        RIGHT JOIN customer c ON o.customer_id = c.customer_id
        WHERE o.order_id IS NULL
        ORDER BY c.name ASC;
""")

,customer_id,name
0,15,ณัฐพล วัฒนกุล
1,19,ดวงใจ แก้วมณี
2,8,นภาพร ใจดี
3,5,ประเสริฐ ทองแท้
4,10,ปราณี สุขใจ
5,16,พิมพ์ใจ อ่อนละมัย
6,11,สมพงษ์ ดีเลิศ
7,17,สายฝน ชื่นใจ
8,13,อนุชา พูลสวัสดิ์


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 10

,customer_id,name
0,15,ณัฐพล วัฒนกุล
1,19,ดวงใจ แก้วมณี
2,8,นภาพร ใจดี
3,5,ประเสริฐ ทองแท้
4,10,ปราณี สุขใจ
5,16,พิมพ์ใจ อ่อนละมัย
6,11,สมพงษ์ ดีเลิศ
7,17,สายฝน ชื่นใจ
8,13,อนุชา พูลสวัสดิ์


### ข้อที่ 11: FULL OUTER JOIN + ความเชื่อมโยงกับ NULL

จงเขียน query ให้ใช้ `FULL OUTER JOIN` ระหว่าง `customer` กับ `orders` ในการหาลูกค้าที่ยังไม่เคยสั่งซื้อเลย และออเดอร์ walk-in ที่ไม่ได้เป็นสมาชิก

In [52]:
#เขียนข้อ 11 โค้ดตรงนี้
run("""
    SELECT c.customer_id, c.name, o.order_id FROM customer c
    FULL JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE c.customer_id IS NULL
      OR o.customer_id IS NULL
    ORDER BY c.customer_id

""")

,customer_id,name,order_id
0,5,ประเสริฐ ทองแท้,<NA>
1,8,นภาพร ใจดี,<NA>
2,10,ปราณี สุขใจ,<NA>
3,11,สมพงษ์ ดีเลิศ,<NA>
4,13,อนุชา พูลสวัสดิ์,<NA>
5,15,ณัฐพล วัฒนกุล,<NA>
6,16,พิมพ์ใจ อ่อนละมัย,<NA>
7,17,สายฝน ชื่นใจ,<NA>
8,19,ดวงใจ แก้วมณี,<NA>
9,<NA>,None,3


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 11

,customer_id,name,order_id
0,5,ประเสริฐ ทองแท้,<NA>
1,8,นภาพร ใจดี,<NA>
2,10,ปราณี สุขใจ,<NA>
3,11,สมพงษ์ ดีเลิศ,<NA>
4,13,อนุชา พูลสวัสดิ์,<NA>
5,15,ณัฐพล วัฒนกุล,<NA>
6,16,พิมพ์ใจ อ่อนละมัย,<NA>
7,17,สายฝน ชื่นใจ,<NA>
8,19,ดวงใจ แก้วมณี,<NA>
9,<NA>,None,3


### ข้อที่ 12: Self-Join

จงเขียน query แบบ **self-join** แสดงชื่อพนักงาน (`employee_name`) คู่กับชื่อหัวหน้า (`manager_name`) ของแต่ละคน (พนักงานที่ไม่มีหัวหน้า เช่น Area Manager ไม่ต้องแสดง) ให้เรียงข้อมูลตาม ชื่อหัวหน้า ก-ฮ และ ชื่อพนักงาน ก-ฮ

In [57]:
run("describe employee")

,column_name,column_type,null,key,default,extra
0,employee_id,INTEGER,NO,PRI,None,None
1,name,VARCHAR,NO,None,None,None
2,position,VARCHAR,NO,None,None,None
3,branch,VARCHAR,NO,None,None,None
4,manager_id,INTEGER,YES,None,None,None
5,hire_date,DATE,NO,None,None,None


In [59]:
#เขียนข้อ 12 โค้ดตรงนี้
run("""
    SELECT
        e.name AS employee_name,
        m.name AS manager_name
    FROM
        employee e
    JOIN
        employee m ON e.manager_id = m.employee_id
    ORDER BY
        m.name ASC,
        e.name ASC

""")

,employee_name,manager_name
0,กิตติ บาริสต้า,ประวิทย์ หัวหน้าสาขา
1,วีระชัย บาริสต้า,ประวิทย์ หัวหน้าสาขา
2,ศศิธร บาริสต้า,ประวิทย์ หัวหน้าสาขา
3,ประวิทย์ หัวหน้าสาขา,สมชาย ผู้จัดการเขต
4,มณีรัตน์ บาริสต้า,สมชาย ผู้จัดการเขต
5,สุนีย์ หัวหน้าสาขา,สมชาย ผู้จัดการเขต
6,ธนกร บาริสต้า,สุนีย์ หัวหน้าสาขา
7,ปิยะดา บาริสต้า,สุนีย์ หัวหน้าสาขา
8,อรทัย บาริสต้า,สุนีย์ หัวหน้าสาขา


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 12

,employee_name,manager_name
0,กิตติ บาริสต้า,ประวิทย์ หัวหน้าสาขา
1,วีระชัย บาริสต้า,ประวิทย์ หัวหน้าสาขา
2,ศศิธร บาริสต้า,ประวิทย์ หัวหน้าสาขา
3,ประวิทย์ หัวหน้าสาขา,สมชาย ผู้จัดการเขต
4,มณีรัตน์ บาริสต้า,สมชาย ผู้จัดการเขต
5,สุนีย์ หัวหน้าสาขา,สมชาย ผู้จัดการเขต
6,ธนกร บาริสต้า,สุนีย์ หัวหน้าสาขา
7,ปิยะดา บาริสต้า,สุนีย์ หัวหน้าสาขา
8,อรทัย บาริสต้า,สุนีย์ หัวหน้าสาขา


### ข้อที่ 13: Multi-condition JOIN

จงเขียน query หา **ออเดอร์ที่มีการสั่งกาแฟราคาแพงกว่า 80 บาท** จัดเรียงตาม order_id จากน้อยไปมาก

In [70]:
#เขียนข้อ 13 โค้ดตรงนี้
run("""
    SELECT
        o.order_id,
        c.name AS customer_name,
        e.name AS employee_name,
        p.product_name,
        p.price
    FROM orders o
    JOIN customer c ON o.customer_id = c.customer_id
    JOIN employee e ON o.employee_id = e.employee_id
    JOIN order_item oi ON o.order_id = oi.order_id
    JOIN product p ON oi.product_id = p.product_id
    WHERE p.category = 'กาแฟ'
      AND p.price > 80
    ORDER BY o.order_id ASC;
""")

,order_id,customer_name,employee_name,product_name,price
0,9,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
1,14,ชัยวัฒน์ รุ่งเรือง,ธนกร บาริสต้า,มอคค่า,90.0
2,18,อรุณี แสงทอง,ศศิธร บาริสต้า,กาแฟเย็นส้มเกลี้ยง,95.0
3,23,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
4,30,ไพโรจน์ เจริญสุข,ปิยะดา บาริสต้า,กาแฟเย็นส้มเกลี้ยง,95.0
5,31,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
6,43,ชัยวัฒน์ รุ่งเรือง,ธนกร บาริสต้า,มอคค่า,90.0
7,49,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 13

,order_id,customer_name,employee_name,product_name,price
0,9,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
1,14,ชัยวัฒน์ รุ่งเรือง,ธนกร บาริสต้า,มอคค่า,90.0
2,18,อรุณี แสงทอง,ศศิธร บาริสต้า,กาแฟเย็นส้มเกลี้ยง,95.0
3,23,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
4,30,ไพโรจน์ เจริญสุข,ปิยะดา บาริสต้า,กาแฟเย็นส้มเกลี้ยง,95.0
5,31,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0
6,43,ชัยวัฒน์ รุ่งเรือง,ธนกร บาริสต้า,มอคค่า,90.0
7,49,สุดา วงศ์ทอง,ธนกร บาริสต้า,มอคค่า,90.0


### ข้อที่ 14: Non-Equi Join

ทางร้านมีตารางอ้างอิงช่วงราคา (สร้างด้วย CTE ชื่อ `price_band(band_name, min_price, max_price)`) ดังนี้

| band_name | min_price | max_price |
|---|---|---|
| ประหยัด | 0 | 59.99 |
| มาตรฐาน | 60 | 85 |
| พรีเมียม | 85.01 | 999 |

จงเขียน query จับคู่สินค้าแต่ละตัวเข้ากับ `band_name` ที่ถูกต้อง โดยใช้ **non-equi join** (เงื่อนไข `ON` ไม่ใช่ `=`)

In [73]:
#เขียนข้อ 14 โค้ดตรงนี้
run("""
WITH price_band (band_name, min_price, max_price) AS (
    VALUES
        ('ประหยัด', 0, 59.99),
        ('มาตรฐาน', 60, 85),
        ('พรีเมียม', 85.01, 999)
)
SELECT
    p.product_name,
    p.price,
    b.band_name
FROM product p
JOIN price_band b
    ON p.price BETWEEN b.min_price AND b.max_price
ORDER BY price ASC

""")

,product_name,price,band_name
0,น้ำเปล่า,20.0,ประหยัด
1,ชามะนาว,55.0,ประหยัด
2,ครัวซองต์เนย,55.0,ประหยัด
3,ชาไทย,60.0,มาตรฐาน
4,ชาดอกเก๊กฮวยเย็น,60.0,มาตรฐาน
5,เอสเปรสโซ่,65.0,มาตรฐาน
6,บราวนี่ช็อกโกแลต,65.0,มาตรฐาน
7,อเมริกาโน่,70.0,มาตรฐาน
8,แซนด์วิชแฮมชีส,70.0,มาตรฐาน
9,เค้กมะพร้าวอ่อน,75.0,มาตรฐาน


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 14

,product_name,price,band_name
0,น้ำเปล่า,20.0,ประหยัด
1,ชามะนาว,55.0,ประหยัด
2,ครัวซองต์เนย,55.0,ประหยัด
3,ชาไทย,60.0,มาตรฐาน
4,ชาดอกเก๊กฮวยเย็น,60.0,มาตรฐาน
5,เอสเปรสโซ่,65.0,มาตรฐาน
6,บราวนี่ช็อกโกแลต,65.0,มาตรฐาน
7,อเมริกาโน่,70.0,มาตรฐาน
8,แซนด์วิชแฮมชีส,70.0,มาตรฐาน
9,เค้กมะพร้าวอ่อน,75.0,มาตรฐาน


---
## Subqueries, CTE & Query Rewriting

### ข้อที่ 15: Scalar Subquery

จงเขียน query หา **สินค้าที่มีราคาสูงกว่าราคาเฉลี่ยของสินค้าทั้งหมด** โดยใช้ scalar subquery

In [78]:
#เขียนข้อ 15 โค้ดตรงนี้
run("""
SELECT product_name, price
FROM product
WHERE price > (SELECT AVG(price) FROM product)
ORDER BY price DESC
""")

,product_name,price
0,กาแฟเย็นส้มเกลี้ยง,95.0
1,มอคค่า,90.0
2,ชาเขียวมัทฉะ,85.0
3,ลาเต้,80.0
4,คาปูชิโน่,80.0
5,เค้กมะพร้าวอ่อน,75.0
6,อเมริกาโน่,70.0
7,แซนด์วิชแฮมชีส,70.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 15

,product_name,price
0,กาแฟเย็นส้มเกลี้ยง,95.0
1,มอคค่า,90.0
2,ชาเขียวมัทฉะ,85.0
3,ลาเต้,80.0
4,คาปูชิโน่,80.0
5,เค้กมะพร้าวอ่อน,75.0
6,อเมริกาโน่,70.0
7,แซนด์วิชแฮมชีส,70.0


### ข้อที่ 16: Scalar Subquery

จงเขียน query หา **สินค้าที่มีราคาต่ำกว่าราคาเฉลี่ยของสินค้าทั้งหมด** โดยใช้ scalar subquery และอยู่ในหมวด "กาแฟ" หรือ "ชา" โดยที่มีราคามากกว่า 55 บาท

In [93]:
#เขียนข้อ 16 โค้ดตรงนี้
run("""
SELECT product_name, price, category FROM product
WHERE category IN ('กาแฟ', 'ชา')
  AND price > 55
  AND price < (SELECT AVG(price) FROM product)
ORDER BY price DESC

""")

,product_name,price,category
0,เอสเปรสโซ่,65.0,กาแฟ
1,ชาไทย,60.0,ชา
2,ชาดอกเก๊กฮวยเย็น,60.0,ชา


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 16

,product_name,price,category
0,บราวนี่ช็อกโกแลต,65.0,เบเกอรี่
1,ชาไทย,60.0,ชา
2,ชาดอกเก๊กฮวยเย็น,60.0,ชา


### ข้อที่ 17: Query Rewriting: Correlated Subquery → JOIN + HAVING

ทางร้านต้องการหา **ลูกค้าที่สั่งซื้ออย่างน้อย 3 ออเดอร์** ให้เรียงจำนวนการครั้งการซื้อจากมากไปน้อย

In [82]:
#เขียนข้อ 17 โค้ดตรงนี้
run("""
  SELECT c.name, COUNT(o.order_id) AS n_orders
  FROM customer c
  JOIN orders o ON c.customer_id = o.customer_id
  GROUP BY c.customer_id, c.name
  HAVING COUNT(o.order_id) >= 3
  ORDER BY n_orders DESC;
""")

,name,n_orders
0,ชัยวัฒน์ รุ่งเรือง,9
1,สุดา วงศ์ทอง,7
2,กมล ศรีสุข,7
3,วิชัย เพชรดี,5
4,อรุณี แสงทอง,4
5,อัครเดช ยิ่งยง,3
6,ธีระพล มั่นคง,3
7,รัตนา เกียรติยศ,3


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 17

,name,n_orders
0,ชัยวัฒน์ รุ่งเรือง,9
1,กมล ศรีสุข,7
2,สุดา วงศ์ทอง,7
3,วิชัย เพชรดี,5
4,อรุณี แสงทอง,4
5,รัตนา เกียรติยศ,3
6,ธีระพล มั่นคง,3
7,อัครเดช ยิ่งยง,3


### ข้อที่ 18: Semi-Join (EXISTS)

จงเขียน query หา **ลูกค้าที่เคยสั่งเครื่องดื่มในหมวด "กาแฟ" อย่างน้อยหนึ่งครั้งหรือไม่** โดยใช้ `EXISTS` (semi-join)

In [86]:
#เขียนข้อ 18 โค้ดตรงนี้
run("""
SELECT c.customer_id, c.name
FROM customer c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_item oi ON o.order_id = oi.order_id
    JOIN product p ON oi.product_id = p.product_id
    WHERE o.customer_id = c.customer_id
      AND p.category = 'กาแฟ'
)
--ORDER BY c.customer_id ASC
""")

,customer_id,name
0,9,ชัยวัฒน์ รุ่งเรือง
1,2,สุดา วงศ์ทอง
2,18,อัครเดช ยิ่งยง
3,1,กมล ศรีสุข
4,4,มาลี บุญมาก
5,12,รัตนา เกียรติยศ
6,20,ไพโรจน์ เจริญสุข
7,3,วิชัย เพชรดี
8,6,อรุณี แสงทอง


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 18

,customer_id,name
0,1,กมล ศรีสุข
1,9,ชัยวัฒน์ รุ่งเรือง
2,4,มาลี บุญมาก
3,12,รัตนา เกียรติยศ
4,3,วิชัย เพชรดี
5,2,สุดา วงศ์ทอง
6,6,อรุณี แสงทอง
7,18,อัครเดช ยิ่งยง
8,20,ไพโรจน์ เจริญสุข


### ข้อที่ 19: CTE + Top-N per Group

จงเขียน query หาว่า **ในแต่ละเมือง ลูกค้าคนใดใช้จ่ายรวมสูงสุด** โดยใช้ CTE ร่วมกับ window function

In [90]:
#เขียนข้อ 19 โค้ดตรงนี้
run("""
WITH customer_spending AS (
    SELECT
        c.city,
        c.name AS customer_name,
        SUM(oi.quantity * oi.unit_price) AS total_spend,
        ROW_NUMBER() OVER (PARTITION BY c.city ORDER BY SUM(oi.quantity * oi.unit_price) DESC) AS rank_num
    FROM customer c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_item oi ON o.order_id = oi.order_id
    GROUP BY c.city, c.customer_id, c.name
)
SELECT
    city,
    customer_name,
    total_spend
FROM customer_spending
WHERE rank_num = 1
ORDER BY total_spend DESC
""")

,city,customer_name,total_spend
0,กรุงเทพฯ,ชัยวัฒน์ รุ่งเรือง,1605.0
1,นนทบุรี,วิชัย เพชรดี,680.0
2,ปทุมธานี,อรุณี แสงทอง,495.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 19

,city,name,total_spend
0,กรุงเทพฯ,ชัยวัฒน์ รุ่งเรือง,1605.0
1,นนทบุรี,วิชัย เพชรดี,680.0
2,ปทุมธานี,อรุณี แสงทอง,495.0


### ข้อที่ 20: Recursive CTE

จงเขียน query แสดง **ลูกทีมทั้งหมด (ทุกระดับชั้น)** ที่อยู่ใต้ "สมชาย ผู้จัดการเขต" (`employee_id = 1`) พร้อมระบุความลึก (`depth`) ของแต่ละคน โดยใช้ `WITH RECURSIVE`

In [92]:
#เขียนข้อ 20 โค้ดตรงนี้
run("""
WITH RECURSIVE subordinates AS (
    SELECT employee_id, name, 1 AS depth
    FROM employee
    WHERE employee_id = 1

    UNION ALL

    SELECT e.employee_id, e.name, s.depth + 1
    FROM employee e
    JOIN subordinates s ON e.manager_id = s.employee_id
)
SELECT * FROM subordinates;


""")

,employee_id,name,depth
0,1,สมชาย ผู้จัดการเขต,1
1,2,สุนีย์ หัวหน้าสาขา,2
2,3,ประวิทย์ หัวหน้าสาขา,2
3,10,มณีรัตน์ บาริสต้า,2
4,4,อรทัย บาริสต้า,3
5,5,ธนกร บาริสต้า,3
6,6,ปิยะดา บาริสต้า,3
7,7,วีระชัย บาริสต้า,3
8,8,ศศิธร บาริสต้า,3
9,9,กิตติ บาริสต้า,3


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 20

,employee_id,name,depth
0,1,สมชาย ผู้จัดการเขต,1
1,2,สุนีย์ หัวหน้าสาขา,2
2,3,ประวิทย์ หัวหน้าสาขา,2
3,10,มณีรัตน์ บาริสต้า,2
4,4,อรทัย บาริสต้า,3
5,5,ธนกร บาริสต้า,3
6,6,ปิยะดา บาริสต้า,3
7,7,วีระชัย บาริสต้า,3
8,8,ศศิธร บาริสต้า,3
9,9,กิตติ บาริสต้า,3


---
## Aggregation & Window Functions

### ข้อที่ 21: GROUP BY + Aggregate Functions

จงเขียน query สรุป **จำนวนออเดอร์ (ไม่นับซ้ำ order_id), จำนวนชิ้นที่ขาย, และยอดขายรวม** แยกตามหมวดหมู่สินค้า (`category`) เรียงจากยอดขายรวมมากไปน้อย

In [94]:
#เขียนข้อ 21 โค้ดตรงนี้
run("""
SELECT
    p.category,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.quantity) AS total_qty,
    SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM product p
JOIN order_item oi ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

""")

,category,n_orders,total_qty,total_revenue
0,กาแฟ,38,54.0,4195.0
1,เบเกอรี่,16,18.0,1140.0
2,ชา,13,16.0,1000.0
3,อื่นๆ,5,5.0,250.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 21

,category,n_orders,total_qty,total_revenue
0,กาแฟ,38,54.0,4195.0
1,เบเกอรี่,16,18.0,1140.0
2,ชา,13,16.0,1000.0
3,อื่นๆ,5,5.0,250.0


### ข้อที่ 22: HAVING

จากข้อก่อนหน้า จงกรองเฉพาะหมวดหมู่ที่มี**ยอดขายรวมมากกว่า 1000 บาท**

In [97]:
#เขียนข้อ 22 โค้ดตรงนี้
run("""
SELECT p.category, SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM product p
JOIN order_item oi ON p.product_id = oi.product_id
GROUP BY p.category
HAVING SUM(oi.quantity * oi.unit_price) > 1000
ORDER BY total_revenue DESC;

""")

,category,total_revenue
0,กาแฟ,4195.0
1,เบเกอรี่,1140.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 22

,category,total_revenue
0,กาแฟ,4195.0
1,เบเกอรี่,1140.0


### ข้อที่ 23: GROUPING SETS / ROLLUP

จงเขียน query สรุปยอดขายแยกตามสาขา (`branch`) และหมวดหมู่ (`category`) พร้อม **subtotal ของแต่ละสาขา** และ **ยอดรวมทั้งหมด**

In [103]:
#เขียนข้อ 23 โค้ดตรงนี้
run("""
SELECT
    o.branch,
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_item oi ON o.order_id = oi.order_id
JOIN product p ON oi.product_id = p.product_id
GROUP BY ROLLUP (o.branch, p.category)
ORDER BY o.branch ASC

""")

,branch,category,revenue
0,รัชดา,กาแฟ,590.0
1,รัชดา,None,590.0
2,สยาม,เบเกอรี่,870.0
3,สยาม,อื่นๆ,230.0
4,สยาม,ชา,420.0
5,สยาม,กาแฟ,3110.0
6,สยาม,None,4630.0
7,เอกมัย,None,1365.0
8,เอกมัย,กาแฟ,495.0
9,เอกมัย,เบเกอรี่,270.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 23

,branch,category,revenue
0,รัชดา,กาแฟ,590.0
1,รัชดา,None,590.0
2,สยาม,กาแฟ,3110.0
3,สยาม,ชา,420.0
4,สยาม,เบเกอรี่,870.0
5,สยาม,อื่นๆ,230.0
6,สยาม,None,4630.0
7,เอกมัย,กาแฟ,495.0
8,เอกมัย,ชา,580.0
9,เอกมัย,เบเกอรี่,270.0


### ข้อที่ 24: Ranking Window Function + Top-N

จงเขียน query หา **สินค้าขายดี 2 อันดับแรก (ตามยอดขายรวม) ในแต่ละหมวดหมู่**

In [107]:
#เขียนข้อ 24 โค้ดตรงนี้
run("""
WITH ranked_sales AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS revenue,
        ROW_NUMBER() OVER (
            PARTITION BY p.category
            ORDER BY SUM(oi.quantity * oi.unit_price) DESC, p.product_name ASC
        ) AS rnk
    FROM product p
    JOIN order_item oi ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT category, product_name, revenue, rnk
FROM ranked_sales
WHERE rnk <= 2
ORDER BY category ASC;
""")

,category,product_name,revenue,rnk
0,กาแฟ,ลาเต้,1440.0,1
1,กาแฟ,มอคค่า,720.0,2
2,ชา,ชาไทย,600.0,1
3,ชา,ชาเขียวมัทฉะ,170.0,2
4,อื่นๆ,แซนด์วิชแฮมชีส,210.0,1
5,อื่นๆ,น้ำเปล่า,40.0,2
6,เบเกอรี่,ครัวซองต์เนย,440.0,1
7,เบเกอรี่,เค้กมะพร้าวอ่อน,375.0,2


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 24

,category,product_name,revenue,rnk
0,กาแฟ,ลาเต้,1440.0,1
1,กาแฟ,มอคค่า,720.0,2
2,ชา,ชาไทย,600.0,1
3,ชา,ชาเขียวมัทฉะ,170.0,2
4,อื่นๆ,แซนด์วิชแฮมชีส,210.0,1
5,อื่นๆ,น้ำเปล่า,40.0,2
6,เบเกอรี่,ครัวซองต์เนย,440.0,1
7,เบเกอรี่,เค้กมะพร้าวอ่อน,375.0,2


### ข้อที่ 25: LAG/LEAD + Running Total

จงเขียน query แสดง**ยอดขายรายเดือนของแต่ละสาขา** พร้อม

- ยอดขายเดือนก่อนหน้า (`LAG`)
- ส่วนต่างเทียบกับเดือนก่อนหน้า (Month-over-Month change)
- ยอดสะสม (running total)

โดยใช้ `PARTITION BY` แยกตามสาขา

In [108]:
#เขียนข้อ 25 โค้ดตรงนี้
run("""
WITH monthly_sales AS (
    SELECT
        branch,
        DATE_TRUNC('month', order_date)::DATE AS month,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_item oi ON o.order_id = oi.order_id
    GROUP BY branch, DATE_TRUNC('month', order_date)::DATE
)
SELECT
    branch,
    month,
    revenue,
    LAG(revenue, 1) OVER (PARTITION BY branch ORDER BY month) AS prev_month_revenue,
    revenue - LAG(revenue, 1) OVER (PARTITION BY branch ORDER BY month) AS mom_change,
    SUM(revenue) OVER (PARTITION BY branch ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM monthly_sales
ORDER BY branch, month;

""")

,branch,month,revenue,prev_month_revenue,mom_change,running_total
0,รัชดา,2024-03-01,70.0,NaN,NaN,70.0
1,รัชดา,2024-04-01,220.0,70.0,150.0,290.0
2,รัชดา,2024-05-01,230.0,220.0,10.0,520.0
3,รัชดา,2024-06-01,70.0,230.0,-160.0,590.0
4,สยาม,2024-02-01,1220.0,NaN,NaN,1220.0
5,สยาม,2024-03-01,790.0,1220.0,-430.0,2010.0
6,สยาม,2024-04-01,785.0,790.0,-5.0,2795.0
7,สยาม,2024-05-01,780.0,785.0,-5.0,3575.0
8,สยาม,2024-06-01,660.0,780.0,-120.0,4235.0
9,สยาม,2024-07-01,395.0,660.0,-265.0,4630.0


In [ ]:
#ตัวอย่างผลลัพธ์ข้อ 25

,branch,month,revenue,prev_month_revenue,mom_change,running_total
0,รัชดา,2024-03-01,70.0,NaN,NaN,70.0
1,รัชดา,2024-04-01,220.0,70.0,150.0,290.0
2,รัชดา,2024-05-01,230.0,220.0,10.0,520.0
3,รัชดา,2024-06-01,70.0,230.0,-160.0,590.0
4,สยาม,2024-02-01,1220.0,NaN,NaN,1220.0
5,สยาม,2024-03-01,790.0,1220.0,-430.0,2010.0
6,สยาม,2024-04-01,785.0,790.0,-5.0,2795.0
7,สยาม,2024-05-01,780.0,785.0,-5.0,3575.0
8,สยาม,2024-06-01,660.0,780.0,-120.0,4235.0
9,สยาม,2024-07-01,395.0,660.0,-265.0,4630.0
